# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.2: Rabi Oscillations, Detuning & Pulse Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/02_rabi_oscillations_and_semiclassical_light.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1** Two-Level Atoms & Optical Bloch Equations
  * **3.2 Rabi Oscillations, Detuning & Pulse Control** $\leftarrow$ *(You are here)*
  * **3.3** The Jaynes-Cummings Hamiltonian & RWA
  * **3.4** The Quantum Rabi Model & Dressed States
  * **3.5** Quantum Collapse and Revival of Atomic Inversion
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7** Electromagnetically Induced Transparency (EIT) & Dark States

---

### Learning Objectives
1. **Coherent Unitary Dynamics:** Solve the Schrödinger equation for a driven two-level atom and derive the generalized Rabi frequency $\Omega_{\text{eff}} = \sqrt{\Omega^2 + \Delta^2}$.
2. **Off-Resonance Dynamics & Amplitude Reduction:** Analyze how laser detuning decreases the maximum excitation probability $P_e^{\text{max}} = \Omega^2 / (\Omega^2 + \Delta^2)$ while accelerating oscillation frequency.
3. **Pulsed Quantum Control:** Implement resonant $\pi$-pulses (complete state inversion) and $\pi/2$-pulses (creation of equal superposition states) via the pulse area theorem $\theta = \int \Omega(t) dt$.
4. **Geometric Bloch Precession:** Visualize state trajectories as rotations around the effective torque vector $\vec{\Omega}_{\text{eff}} = (\Omega, 0, -\Delta)^T$ on the Bloch sphere.
5. **Ramsey Interferometry:** Simulate the two-pulse Ramsey sequence ($\pi/2 - T - \pi/2$), observe narrow Ramsey fringes with linewidth $\delta\nu \sim 1/(2T)$, and demonstrate frequency metrology beyond transit-time limits.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Coherent Two-Level Atom Dynamics in the Rotating Frame

In Notebook 3.1, we studied steady-state dissipative dynamics where continuous-wave lasers reached an equilibrium population limit $\rho_{ee} \le 0.5$. On timescales much shorter than the atomic lifetime ($t \ll T_1, T_2$), dissipation can be completely neglected, and the dynamics are strictly **coherent and unitary**.

In the rotating wave approximation (RWA), the effective Hamiltonian in the laser rotating frame is:
$$\hat{H}_{\text{RWA}} = \frac{\hbar}{2} \begin{pmatrix} -\Delta & \Omega \\ \Omega & \Delta \end{pmatrix} = -\frac{\hbar \Delta}{2} \hat{\sigma}_z + \frac{\hbar \Omega}{2} \hat{\sigma}_x$$
where:
* $\Delta = \omega_L - \omega_0$ is the laser detuning from atomic transition frequency $\omega_0$.
* $\Omega = -\vec{d}_{eg}\cdot\vec{E}_0 / \hbar$ is the bare Rabi frequency.

### Analytical Solution of the Time-Dependent Schrödinger Equation
For an atom initially in the ground state $|\psi(0)\rangle = |g\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$, the state vector at time $t$ is:
$$|\psi(t)\rangle = \exp\left( -i \frac{\hat{H}_{\text{RWA}} t}{\hbar} \right) |g\rangle = c_g(t)|g\rangle + c_e(t)|e\rangle$$

Using the matrix exponential identity for Pauli matrices $\exp(-i \frac{\vec{a}\cdot\vec{\sigma}}{2}) = \cos\frac{|\vec{a}|}{2} \hat{I} - i \frac{\vec{a}\cdot\vec{\sigma}}{|\vec{a}|} \sin\frac{|\vec{a}|}{2}$, we obtain the exact probability amplitudes:
$$c_e(t) = -i \frac{\Omega}{\Omega_{\text{eff}}} \sin\left( \frac{\Omega_{\text{eff}} t}{2} \right) \exp\left( -i \frac{\Delta t}{2} \right)$$
$$c_g(t) = \left[ \cos\left( \frac{\Omega_{\text{eff}} t}{2} \right) + i \frac{\Delta}{\Omega_{\text{eff}}} \sin\left( \frac{\Omega_{\text{eff}} t}{2} \right) \right] \exp\left( -i \frac{\Delta t}{2} \right)$$
where the **generalized (effective) Rabi frequency** is:
$$\Omega_{\text{eff}} = \sqrt{\Omega^2 + \Delta^2}$$

### The Population Dynamics:
The probability of finding the atom in the excited state $|e\rangle$ is the celebrated **Rabi flopping formula**:
$$P_e(t) = |c_e(t)|^2 = \frac{\Omega^2}{\Omega^2 + \Delta^2} \sin^2\left( \frac{\Omega_{\text{eff}} t}{2} \right) = \frac{\Omega^2}{2\Omega_{\text{eff}}^2} \left[ 1 - \cos(\Omega_{\text{eff}} t) \right]$$

### Key Physical Features:
1. **On Resonance ($\Delta = 0$):**
   $$\Omega_{\text{eff}} = \Omega \implies P_e(t) = \sin^2\left(\frac{\Omega t}{2}\right)$$
   The atom undergoes full sinusoidal flopping between $|g\rangle$ and $|e\rangle$ with 100% maximum inversion ($P_e = 1.0$) at times $t = (2k+1)\pi/\Omega$.
2. **Off Resonance ($\Delta \ne 0$):**
   * The oscillation frequency **increases** ($\Omega_{\text{eff}} > \Omega$).
   * The maximum excitation amplitude is **strictly diminished**:
     $$P_e^{\text{max}} = \frac{\Omega^2}{\Omega^2 + \Delta^2} < 1.0$$
     When $\Delta \gg \Omega$, $P_e^{\text{max}} \sim (\Omega/\Delta)^2 \to 0$: the atom is barely excited, only acquiring a dispersive phase shift.


---

## 2. The Pulse Area Theorem & Precision Quantum Gates

When a time-dependent laser pulse $\Omega(t)$ is applied on resonance ($\Delta = 0$), the total rotation angle on the Bloch sphere is given by the **pulse area**:
$$\theta = \int_{-\infty}^\infty \Omega(t) \, dt$$

The evolution operator is a pure rotation around the $x$-axis by angle $\theta$:
$$\hat{R}_x(\theta) = \exp\left( -i \frac{\theta}{2} \hat{\sigma}_x \right) = \begin{pmatrix} \cos(\theta/2) & -i\sin(\theta/2) \\ -i\sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$

### Canonical Quantum Gates:
* **The $\pi$-Pulse ($\theta = \pi$):**
  $$\hat{R}_x(\pi) |g\rangle = -i |e\rangle \implies P_e = 1.0$$
  Inverts the atomic population completely from ground to excited state. This acts as a quantum **NOT gate** (or bit-flip operator $\hat{\sigma}_x$).
* **The $\pi/2$-Pulse ($\theta = \pi/2$):**
  $$\hat{R}_x(\pi/2) |g\rangle = \frac{|g\rangle - i |e\rangle}{\sqrt{2}}$$
  Creates an equal, coherent superposition of ground and excited states, placing the Bloch vector exactly on the equator ($w = 0, v = -1, u = 0$).

---

## 3. Ramsey Separated Oscillatory Fields Interferometry

In 1949, **Norman Ramsey** devised an ingenious method to overcome the limitations of power broadening and transit-time broadening in molecular and atomic spectroscopy.

Rather than continuously illuminating an atom for a long time, the **Ramsey sequence** consists of:
1. **Pulse 1:** A resonant $\pi/2$-pulse of duration $\tau$ that creates an equatorial superposition state:
   $$|\psi_1\rangle = \frac{|g\rangle - i|e\rangle}{\sqrt{2}}$$
2. **Free Precession (Dark Time $T$):** The laser is turned off ($\Omega = 0$) for duration $T \gg \tau$. During this dark time, the atomic dipole precesses in the rotating frame at the detuning frequency $\Delta = \omega_L - \omega_0$:
   $$|\psi_2\rangle = \frac{|g\rangle - i e^{-i \Delta T} |e\rangle}{\sqrt{2}}$$
3. **Pulse 2:** A second $\pi/2$-pulse with relative laser phase $\phi$ recombines the amplitudes.

### The Ramsey Fringe Pattern:
The resulting excited-state population after the second pulse is:
$$P_e(\Delta) = \cos^2\left( \frac{\Delta T - \phi}{2} \right) = \frac{1 + \cos(\Delta T - \phi)}{2}$$

The central fringe has a full-width at half-maximum (FWHM) of:
$$\delta\nu_{\text{Ramsey}} \approx \frac{1}{2T}$$
Because the dark time $T$ can be extended to several seconds (e.g. in laser-cooled atomic fountains), this produces ultra-narrow spectral features with quality factors $Q = \omega_0 / \delta\omega > 10^{15}$, serving as the physical foundation of **cesium and optical lattice atomic clocks**!


In [ ]:
# Implementation: Analytical and Numerical Coherent Dynamics Solver

class CoherentTwoLevelSimulator:
    """
    Simulates coherent unitary dynamics, arbitrary pulse sequences,
    and Ramsey interferometry for a two-level atom.
    """
    def __init__(self, omega=1.0, delta=0.0):
        self.omega = float(omega)
        self.delta = float(delta)
        
    @property
    def omega_eff(self):
        """Effective generalized Rabi frequency sqrt(Omega^2 + Delta^2)."""
        return np.sqrt(self.omega**2 + self.delta**2)
        
    def analytical_pe(self, t):
        """Exact excited state population Pe(t) starting from |g>."""
        if self.omega_eff == 0:
            return np.zeros_like(t)
        return (self.omega**2 / self.omega_eff**2) * np.sin(0.5 * self.omega_eff * t)**2
        
    def hamiltonian(self, omega=None, delta=None, phi=0.0):
        """Constructs the 2x2 RWA Hamiltonian in energy basis (|e>=[1,0], |g>=[0,1])."""
        om = self.omega if omega is None else omega
        d = self.delta if delta is None else delta
        # H = -hbar*delta/2 * sigma_z + hbar*omega/2 * (cos(phi)*sigma_x + sin(phi)*sigma_y)
        h11 = -0.5 * d
        h22 =  0.5 * d
        h12 =  0.5 * om * np.exp(-1j * phi)
        h21 =  0.5 * om * np.exp(1j * phi)
        return np.array([[h11, h12], [h21, h22]], dtype=complex)
        
    def evolve_state(self, psi0, t_duration, omega=None, delta=None, phi=0.0):
        """Propagates state vector psi0 under constant Hamiltonian for duration t_duration."""
        H = self.hamiltonian(omega=omega, delta=delta, phi=phi)
        U = la.expm(-1j * H * t_duration)
        return U @ psi0
        
    def simulate_ramsey(self, delta_range, T_dark, tau_pulse=0.05, phi_phase=0.0):
        """
        Simulates the Ramsey sequence: pi/2 pulse -> dark time T -> pi/2 pulse.
        """
        # pi/2 pulse requires Omega * tau = pi/2 -> Omega = pi / (2 * tau)
        omega_pi2 = np.pi / (2.0 * tau_pulse)
        pe_results = []
        
        for d in delta_range:
            psi = np.array([0.0, 1.0], dtype=complex)  # Start in |g>
            # Step 1: First pi/2 pulse along x (phi=0)
            psi = self.evolve_state(psi, tau_pulse, omega=omega_pi2, delta=d, phi=0.0)
            # Step 2: Dark time T (Omega = 0, precession at detuning d)
            psi = self.evolve_state(psi, T_dark, omega=0.0, delta=d, phi=0.0)
            # Step 3: Second pi/2 pulse with phase phi_phase
            psi = self.evolve_state(psi, tau_pulse, omega=omega_pi2, delta=d, phi=phi_phase)
            # Measure Pe = |<e|psi>|^2
            pe_results.append(np.abs(psi[0])**2)
            
        return np.array(pe_results)

# Verification Benchmark: Compare numerical state evolution with exact analytical formula
sim_test = CoherentTwoLevelSimulator(omega=2.5, delta=1.5)
t_test = np.linspace(0.0, 10.0, 1000)
pe_analytical = sim_test.analytical_pe(t_test)

# Numerical check via matrix exponential
psi_init = np.array([0.0, 1.0], dtype=complex)
pe_numerical = []
for t in t_test:
    psi_t = sim_test.evolve_state(psi_init, t)
    pe_numerical.append(np.abs(psi_t[0])**2)
pe_numerical = np.array(pe_numerical)

max_error = np.max(np.abs(pe_analytical - pe_numerical))
print("--- Coherent Two-Level Simulator Verification ---")
print(f"Generalized Rabi Frequency Omega_eff: {sim_test.omega_eff:.4f}")
print(f"Maximum Inversion Amplitude Pe_max:   {sim_test.omega**2 / sim_test.omega_eff**2:.4f}")
print(f"Max Numerical vs. Analytical Error:   {max_error:.3e}")
assert max_error < 1e-12, "Coherent unitary evolution benchmark failed!"
print("Verification Passed: Numerical propagator matches exact analytical Rabi solution to machine precision!")


In [ ]:
# Publication-Quality 4-Panel Visualization: Rabi Flopping, Bloch Precession, Pulse Control, and Ramsey Fringes

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: On- vs. Off-Resonance Rabi Flopping
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

t_span = np.linspace(0.0, 12.0, 800)
omega_base = 2.0
detunings = [0.0, 1.0, 2.0, 4.0]
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']

for d, c in zip(detunings, colors):
    sim = CoherentTwoLevelSimulator(omega=omega_base, delta=d)
    pe_curve = sim.analytical_pe(t_span)
    om_eff = sim.omega_eff
    p_max = omega_base**2 / om_eff**2
    ax1.plot(t_span, pe_curve, color=c, label=rf"$\Delta = {d:.1f}\Omega$ ($\Omega_{{\mathrm{{eff}}}} = {om_eff:.2f}$, $P_e^{{\mathrm{{max}}}} = {p_max:.2f}$)")

ax1.set_xlabel(r"Interaction Time ($\Omega t$)")
ax1.set_ylabel(r"Excited State Population $P_e(t)$")
ax1.set_title(r"Rabi Flopping: Amplitude Reduction & Frequency Shift")
ax1.set_ylim(-0.02, 1.08)
ax1.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 2: Bloch Sphere Trajectories: Resonant vs. Off-Resonant Precession
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2, projection='3d')

# Render translucent Bloch sphere
phi_g, theta_g = np.mgrid[0.0:2.0*np.pi:40j, 0.0:np.pi:20j]
xs = np.sin(theta_g) * np.cos(phi_g)
ys = np.sin(theta_g) * np.sin(phi_g)
zs = np.cos(theta_g)
ax2.plot_surface(xs, ys, zs, color='silver', alpha=0.12, edgecolor='none')
ax2.plot_wireframe(xs, ys, zs, color='gray', alpha=0.2, rstride=5, cstride=5, linewidth=0.5)

# Render axes
ax2.plot([-1.1, 1.1], [0, 0], [0, 0], 'k-', lw=0.8, alpha=0.4)
ax2.plot([0, 0], [-1.1, 1.1], [0, 0], 'k-', lw=0.8, alpha=0.4)
ax2.plot([0, 0], [0, 0], [-1.1, 1.1], 'k-', lw=0.8, alpha=0.4)

# Resonant trajectory (Great circle from south pole to north pole)
t_orb = np.linspace(0.0, 2*np.pi/omega_base, 300)
sim_res = CoherentTwoLevelSimulator(omega=omega_base, delta=0.0)
u_res, v_res, w_res = [], [], []
psi_g = np.array([0.0, 1.0], dtype=complex)
for t in t_orb:
    psi_t = sim_res.evolve_state(psi_g, t)
    # Bloch components: u = 2*Re(rho_ge), v = 2*Im(rho_ge), w = |ce|^2 - |cg|^2
    u_res.append(2.0 * np.real(psi_t[1].conj() * psi_t[0]))
    v_res.append(2.0 * np.imag(psi_t[1].conj() * psi_t[0]))
    w_res.append(np.abs(psi_t[0])**2 - np.abs(psi_t[1])**2)

# Off-resonant trajectory (Cone of precession)
sim_off = CoherentTwoLevelSimulator(omega=omega_base, delta=2.0)
t_off = np.linspace(0.0, 2*np.pi/sim_off.omega_eff, 300)
u_off, v_off, w_off = [], [], []
for t in t_off:
    psi_t = sim_off.evolve_state(psi_g, t)
    u_off.append(2.0 * np.real(psi_t[1].conj() * psi_t[0]))
    v_off.append(2.0 * np.imag(psi_t[1].conj() * psi_t[0]))
    w_off.append(np.abs(psi_t[0])**2 - np.abs(psi_t[1])**2)

ax2.plot(u_res, v_res, w_res, 'b-', lw=2.5, label=r"Resonant $\Delta = 0$ (Great Circle)")
ax2.plot(u_off, v_off, w_off, 'r--', lw=2.5, label=r"Off-Resonant $\Delta = 2\Omega$ (Cone Precession)")
ax2.plot([0], [0], [-1], 'ko', markersize=8, label=r"Ground State $|g\rangle$")
ax2.plot([0], [0], [1], 'go', markersize=8, label=r"Excited State $|e\rangle$")

ax2.set_xlabel("$u$ (x)", labelpad=6)
ax2.set_ylabel("$v$ (y)", labelpad=6)
ax2.set_zlabel("$w$ (z)", labelpad=6)
ax2.set_title("Bloch Sphere Precession Dynamics")
ax2.legend(loc="upper left", fontsize=8, frameon=True)
ax2.view_init(elev=20, azim=55)

# =============================================================================
# Panel 3: Precision Pulse Control: pi and pi/2 Gates
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

# Pulse sequence demonstration: pi/2 pulse -> wait -> pi pulse
sim_ctrl = CoherentTwoLevelSimulator(omega=1.0, delta=0.0)
tau_pi = np.pi / 1.0  # Duration of pi pulse
tau_pi2 = tau_pi / 2.0

t_pulse = np.linspace(0.0, 2.5 * tau_pi, 1000)
pe_pulse_history = []
w_pulse_history = []
psi_curr = np.array([0.0, 1.0], dtype=complex)

for t in t_pulse:
    # Evolve under constant Omega
    psi_t = sim_ctrl.evolve_state(np.array([0.0, 1.0], dtype=complex), t)
    pe_val = np.abs(psi_t[0])**2
    pe_pulse_history.append(pe_val)
    w_pulse_history.append(pe_val - (1.0 - pe_val))

ax3.plot(t_pulse / np.pi, pe_pulse_history, 'b-', label=r"Population $P_e(t)$")
ax3.plot(t_pulse / np.pi, w_pulse_history, 'g--', label=r"Inversion $w(t)$")

ax3.axvline(0.5, color='orange', linestyle=':', lw=2, label=r"$\pi/2$-Pulse ($P_e = 0.5$, Superposition)")
ax3.axvline(1.0, color='red', linestyle=':', lw=2, label=r"$\pi$-Pulse ($P_e = 1.0$, Complete Inversion)")
ax3.axvline(2.0, color='purple', linestyle=':', lw=2, label=r"$2\pi$-Pulse ($P_e = 0.0$, Identity Gate)")

ax3.set_xlabel(r"Pulse Area $\theta / \pi = \Omega t / \pi$")
ax3.set_ylabel("State Metric")
ax3.set_title(r"Pulse Area Theorem: Unitary Quantum Gates")
ax3.set_ylim(-1.05, 1.08)
ax3.legend(loc="lower left", frameon=True)

# =============================================================================
# Panel 4: Ramsey Interferometry Fringes (Separated Oscillatory Fields)
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

delta_ramsey = np.linspace(-6.0, 6.0, 600)
T_dark_list = [2.0, 5.0, 10.0]
ramsey_colors = ['#1f77b4', '#2ca02c', '#d62728']

for T_d, col in zip(T_dark_list, ramsey_colors):
    pe_ramsey = sim_ctrl.simulate_ramsey(delta_ramsey, T_dark=T_d, tau_pulse=0.05)
    fwhm_ramsey = np.pi / T_d
    ax4.plot(delta_ramsey, pe_ramsey, color=col, label=rf"Dark Time $T = {T_d:.0f}$ (Linewidth $\approx {fwhm_ramsey:.2f}$)")

ax4.set_xlabel(r"Detuning $\Delta = \omega_L - \omega_0$")
ax4.set_ylabel(r"Final Transition Probability $P_e$")
ax4.set_title(r"Ramsey Interferometry: Linewidth Narrowing $\delta\nu \propto 1/T$")
ax4.set_ylim(-0.02, 1.08)
ax4.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Does Ramsey Interferometry Outperform Continuous Rabi Driving in Precision Clocks?**
>
> In continuous Rabi driving, trying to increase resolution by driving weakly for a long time causes severe **power broadening** and makes the transition vulnerable to laser intensity fluctuations $\delta\Omega$. Furthermore, atoms moving through a continuous laser beam experience **transit-time broadening** ($\delta\omega \sim 1/\tau_{\text{transit}}$).
>
> In the **Ramsey technique**, the laser field is applied only during two brief $\pi/2$-pulses separated by a field-free "dark" evolution time $T$:
> 1. During the dark time $T$, the atoms are in a free-flying vacuum environment with **zero light shifts (AC Stark shifts)** and zero power broadening.
> 2. The central fringe linewidth is set purely by the interrogation time $\delta\nu \approx 1/(2T)$, completely independent of laser power!
> 3. By launching laser-cooled atoms vertically in an "atomic fountain" where $T \sim 1\text{ second}$, atomic clocks achieve fractional frequency uncertainties $< 10^{-16}$, defining the SI second with unprecedented precision.


In [ ]:
# Interactive Exploration: Ramsey Interferometer with Relative Phase Control

def plot_interactive_ramsey(T_dark=5.0, phase_deg=0.0):
    """
    Simulates the Ramsey interference fringes as a function of detuning
    and relative laser phase shift between the two pulses.
    """
    phi_rad = np.radians(phase_deg)
    delta_scan = np.linspace(-5.0, 5.0, 500)
    
    sim = CoherentTwoLevelSimulator(omega=1.0, delta=0.0)
    pe_curve = sim.simulate_ramsey(delta_scan, T_dark=T_dark, tau_pulse=0.05, phi_phase=phi_rad)
    
    # Analytical prediction: Pe(Delta) = cos^2((Delta*T - phi)/2)
    pe_analytical = np.cos(0.5 * (delta_scan * T_dark - phi_rad))**2
    
    fig, (ax_fringe, ax_bloch) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Fringe profile
    ax_fringe.plot(delta_scan, pe_curve, 'b-', lw=2.2, label="Numerical Pulse Simulation")
    ax_fringe.plot(delta_scan, pe_analytical, 'r--', lw=1.5, alpha=0.7, label=r"Analytical $\cos^2[(\Delta T - \phi)/2]$")
    ax_fringe.axvline(phi_rad / T_dark, color='k', linestyle=':', label=rf"Central Fringe Peak ($\Delta = \phi/T = {phi_rad/T_dark:.2f}$)")
    ax_fringe.set_xlabel(r"Detuning $\Delta$")
    ax_fringe.set_ylabel(r"Excited State Population $P_e$")
    ax_fringe.set_title(rf"Ramsey Fringes ($T={T_dark:.1f}$, Phase Shift $\phi={phase_deg:.0f}^\circ$)")
    ax_fringe.set_ylim(-0.05, 1.05)
    ax_fringe.legend(frameon=True)
    
    # Bloch vector phase angle during dark time on resonance (Delta = 0)
    theta_angles = np.linspace(0, 2*np.pi, 200)
    ax_bloch.plot(np.cos(theta_angles), np.sin(theta_angles), 'gray', linestyle='--', alpha=0.5)
    
    # Vector after first pi/2 pulse
    ax_bloch.arrow(0, 0, 0, -0.9, head_width=0.08, head_length=0.08, fc='orange', ec='orange', lw=2, label="After 1st Pulse")
    # Vector after phase shift phi
    v_x = -np.sin(phi_rad) * 0.9
    v_y = -np.cos(phi_rad) * 0.9
    ax_bloch.arrow(0, 0, v_x, v_y, head_width=0.08, head_length=0.08, fc='red', ec='red', lw=2, label=r"After Phase Shift $\phi$")
    
    ax_bloch.set_xlim(-1.2, 1.2)
    ax_bloch.set_ylim(-1.2, 1.2)
    ax_bloch.set_aspect('equal')
    ax_bloch.set_xlabel("Equatorial $u$ (Dispersion)")
    ax_bloch.set_ylabel("Equatorial $v$ (Absorption)")
    ax_bloch.set_title("Equatorial Bloch Vector Phase Evolution")
    ax_bloch.legend(loc="upper right", frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_ramsey,
        T_dark=widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5, description="Dark Time $T$:"),
        phase_deg=widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=15.0, description=r"Phase $\phi^\circ$:")
    )
else:
    print("Static exploration demonstration (default parameters: T = 5.0, phase = 90 deg):")
    plot_interactive_ramsey(T_dark=5.0, phase_deg=90.0)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Detuned $\pi$-Pulse Fidelity
In quantum information processing, a laser frequency jitter $\delta\Delta$ causes an intended resonant $\pi$-pulse to be slightly off-resonance.
For a nominal $\pi$-pulse duration $\tau = \pi/\Omega$, derive the population transfer fidelity $\mathcal{F} = P_e(\tau)$ as a function of small detuning $\epsilon = \Delta/\Omega \ll 1$.

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

The off-resonant Rabi formula gives:
$$P_e(t) = \frac{\Omega^2}{\Omega^2 + \Delta^2} \sin^2\left( \frac{\sqrt{\Omega^2 + \Delta^2} t}{2} \right)$$
Setting $t = \tau = \pi/\Omega$ and $\Delta = \epsilon \Omega$:
$$\frac{\Omega^2}{\Omega^2 + \Delta^2} = \frac{1}{1 + \epsilon^2} \approx 1 - \epsilon^2$$
The argument of the sine function is:
$$\frac{\Omega_{\text{eff}} \tau}{2} = \frac{\Omega \sqrt{1 + \epsilon^2}}{2} \frac{\pi}{\Omega} = \frac{\pi}{2} \sqrt{1 + \epsilon^2} \approx \frac{\pi}{2} \left( 1 + \frac{\epsilon^2}{2} \right) = \frac{\pi}{2} + \frac{\pi \epsilon^2}{4}$$
Using $\sin(\frac{\pi}{2} + x) = \cos(x) \approx 1 - \frac{x^2}{2}$:
$$\sin^2\left( \frac{\Omega_{\text{eff}} \tau}{2} \right) \approx \left[ 1 - \frac{1}{2} \left( \frac{\pi \epsilon^2}{4} \right)^2 \right]^2 \approx 1 - \mathcal{O}(\epsilon^4)$$
Multiplying the prefactor and trigonometric term to second order in $\epsilon$:
$$P_e(\tau) \approx (1 - \epsilon^2) \times 1 = 1 - \left( \frac{\Delta}{\Omega} \right)^2$$
Thus, the gate error scales **quadratically** with detuning: $1 - \mathcal{F} \approx (\Delta/\Omega)^2$.
</details>

---

### Challenge 2: Numerical Verification of Detuned Gate Infidelity
Simulate an intended $\pi$-pulse with detuning $\Delta = 0.1\Omega$. Verify that the population error $1 - P_e(\pi/\Omega)$ matches the analytical prediction $(\Delta/\Omega)^2 = 0.01$ to leading order.


In [ ]:
# Numerical Verification of Challenge 2: Detuned Pi-Pulse Infidelity

om_nominal = 1.0
t_pi = np.pi / om_nominal
epsilon = 0.1
delta_test = epsilon * om_nominal

sim_detuned = CoherentTwoLevelSimulator(omega=om_nominal, delta=delta_test)
pe_numerical = sim_detuned.analytical_pe(np.array([t_pi]))[0]

actual_infidelity = 1.0 - pe_numerical
predicted_infidelity = epsilon**2

print(f"--- Detuned Pi-Pulse Infidelity Benchmark (epsilon = {epsilon}) ---")
print(f"Numerical Excited Population Pe:  {pe_numerical:.6f}")
print(f"Actual Infidelity (1 - Pe):       {actual_infidelity:.6f}")
print(f"Second-Order Prediction (eps^2):  {predicted_infidelity:.6f}")
print(f"Relative Discrepancy:             {abs(actual_infidelity - predicted_infidelity) / predicted_infidelity * 100:.2f}%")

assert abs(actual_infidelity - predicted_infidelity) < 0.001, "Detuned pi-pulse infidelity benchmark failed!"
print("Verification Passed: Gate infidelity matches theoretical perturbation formula to < 0.1% accuracy!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Formula | Physical Meaning |
| :--- | :--- | :--- |
| **Effective Rabi Frequency** | $\Omega_{\text{eff}} = \sqrt{\Omega^2 + \Delta^2}$ | Accelerated oscillation frequency under detuning |
| **Rabi Flopping Formula** | $P_e(t) = \frac{\Omega^2}{\Omega_{\text{eff}}^2} \sin^2(\frac{\Omega_{\text{eff}} t}{2})$ | Time-dependent excitation probability |
| **Max Off-Resonant Inversion** | $P_e^{\text{max}} = \frac{\Omega^2}{\Omega^2 + \Delta^2}$ | Detuning suppresses excitation amplitude |
| **Pulse Area Theorem** | $\theta = \int \Omega(t) dt$ | Angle of unitary rotation on the Bloch sphere |
| **Ramsey Fringe Formula** | $P_e(\Delta) = \frac{1 + \cos(\Delta T - \phi)}{2}$ | Interference fringes with linewidth $\delta\nu \approx 1/(2T)$ |

---

### Further Reading & Foundational Literature
* **I. I. Rabi**, *Space Quantization in a Gyrating Magnetic Field*, Phys. Rev. 51, 652 (1937).
* **N. F. Ramsey**, *A Molecular Beam Resonance Method with Separated Oscillating Fields*, Phys. Rev. 78, 695 (1950) [Nobel Prize 1989].
* **C. J. Foot**, *Atomic Physics*, Oxford Master Series in Physics (2005), Chapters 7 & 8.

---

### Next Notebook in Module 3
In **Notebook 3.3: The Jaynes-Cummings Hamiltonian & RWA**, we cross the threshold from **semiclassical** light (where the field is a classical $c$-number $\vec{E}_0\cos\omega t$) to **fully quantized Cavity QED**:
* Quantizing the cavity electromagnetic mode with creation and annihilation operators $\hat{a}^\dagger, \hat{a}$.
* Deriving the Jaynes-Cummings Hamiltonian $\hat{H}_{\text{JC}} = \hbar\omega_c \hat{a}^\dagger\hat{a} + \frac{\hbar\omega_a}{2}\hat{\sigma}_z + \hbar g (\hat{a}\hat{\sigma}_+ + \hat{a}^\dagger\hat{\sigma}_-)$.
* Analyzing block-diagonal $2 \times 2$ invariant subspaces and the fundamental single-photon Rabi frequency $2g\sqrt{n+1}$.
